In [ ]:
import sys
from pathlib import Path
for _root in [Path.cwd(), *Path.cwd().parents]:
    if (_root / "paths.py").exists():
        sys.path.insert(0, str(_root))
        break
else:
    raise RuntimeError(
        "Could not find GPT5 project root (paths.py). Run Jupyter with cwd GPT5 or GPT5/notebooks."
    )
import paths

## Initiate GPT5 and Trainees Labels Concordance Rate


In [4]:
import pandas as pd

# Load the model predictions
output_path = paths.RELEVANCY / "gpt5-relevancy-combined-dec-12.csv"
gpt5 = pd.read_csv(output_path)
gpt5.columns


original_path = paths.DATA / "Centaur_Lab_First_Round_COMPLETE_RAW.csv"
trainee = pd.read_csv(original_path)
trainee.columns

Index(['Origin', 'q1', 'q2', 'q3', 'q4', 'q5', 'q6', 'q7', 'q8', 'q9', 'q10',
       'q11', 'q12', 'q13', 'q14', 'q15', 'q16', 'q17', 'q18', 'q19', 'q20',
       'ID_corr', 'sentence_number_corr', 'answer_corr', 'data_source_corr',
       'REMOVED_Sentences', 'sentence_number_df3', 'step1_excerpts',
       'question_options', 'Filtered_Sentences', 'New_Sentences'],
      dtype='object')

In [5]:
gpt5.columns

Index(['Unnamed: 0', 'ID_corr', 'centaur_question_corr', 'answer_corr',
       'data_source_corr', 'majority_vote', 'run1_response', 'run2_response',
       'run3_response', 'question_options', 'label_1', 'label_2', 'label_3',
       'label_4', 'label_5', 'label_6', 'label_7', 'label_8', 'label_9',
       'label_10', 'label_11', 'label_12', 'label_13', 'label_14', 'label_15',
       'label_16', 'label_17', 'label_18', 'label_19', 'label_20', 'label_21'],
      dtype='object')

In [6]:
cols_trainee = [f"q{i}" for i in range(1, 21)]

def transform_trainee(value):
    if pd.isna(value):
        return value
    
    val_str = str(value).strip().upper()
    
    # Check if numeric (including floats stored as strings)
    try:
        float(value)
        return "high relevance"
    except ValueError:
        pass
    
    if val_str == "REMOVED":
        return "low relevance"
    else:
        return value

trainee[cols_trainee] = trainee[cols_trainee].apply(lambda col: col.map(transform_trainee))

In [7]:
trainee[["Origin", 'data_source_corr'] + cols_trainee].head(10)

,Origin,data_source_corr,q1,q2,q3,q4,q5,q6,q7,q8,...,q11,q12,q13,q14,q15,q16,q17,q18,q19,q20
0,ID0002,jama,high relevance,high relevance,low relevance,high relevance,low relevance,high relevance,low relevance,low relevance,...,high relevance,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ID0003,medxpert,high relevance,high relevance,low relevance,high relevance,high relevance,high relevance,low relevance,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ID0007,medbullets,low relevance,high relevance,high relevance,high relevance,high relevance,low relevance,high relevance,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ID0009,jama,low relevance,high relevance,high relevance,high relevance,high relevance,high relevance,low relevance,low relevance,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ID0010,medxpert,high relevance,low relevance,low relevance,low relevance,low relevance,low relevance,high relevance,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,ID0013,mmlu,high relevance,high relevance,high relevance,low relevance,low relevance,high relevance,high relevance,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,ID0015,jama,high relevance,high relevance,low relevance,high relevance,low relevance,high relevance,low relevance,low relevance,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,ID0016,jama,high relevance,low relevance,high relevance,low relevance,low relevance,low relevance,low relevance,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,ID0017,jama,high relevance,high relevance,high relevance,high relevance,low relevance,low relevance,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,ID0018,jama,high relevance,high relevance,low relevance,low relevance,high relevance,high relevance,high relevance,high relevance,...,high relevance,high relevance,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
cols_gpt5 = [f"label_{i}" for i in range(1, 22)]

gpt5_sorted = gpt5.sort_values(by="ID_corr")

gpt5_renamed = gpt5_sorted.rename(
    columns=lambda c: c.replace("label_", "q") if c.startswith("label_") else c
)

gpt5_renamed[["ID_corr"] + [f"q{i}" for i in range(1, 22)]].head(10)

,ID_corr,q1,q2,q3,q4,q5,q6,q7,q8,q9,...,q12,q13,q14,q15,q16,q17,q18,q19,q20,q21
0,ID0002,High Relevance,High Relevance,High Relevance,High Relevance,Low Relevance,High Relevance,Low Relevance,Low Relevance,Low Relevance,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ID0003,High Relevance,Low Relevance,Irrelevant,Irrelevant,Low Relevance,High Relevance,High Relevance,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ID0007,High Relevance,Low Relevance,High Relevance,High Relevance,High Relevance,Low Relevance,High Relevance,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ID0009,High Relevance,High Relevance,High Relevance,Low Relevance,High Relevance,High Relevance,Irrelevant,High Relevance,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ID0010,High Relevance,Low Relevance,Irrelevant,Irrelevant,Low Relevance,Low Relevance,High Relevance,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,ID0013,High Relevance,High Relevance,High Relevance,Irrelevant,Irrelevant,Irrelevant,Low Relevance,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,ID0015,High Relevance,Low Relevance,Low Relevance,High Relevance,Low Relevance,High Relevance,High Relevance,Low Relevance,High Relevance,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,ID0016,High Relevance,Low Relevance,High Relevance,High Relevance,High Relevance,High Relevance,High Relevance,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,ID0017,Low Relevance,Low Relevance,Low Relevance,High Relevance,Low Relevance,Low Relevance,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,ID0018,High Relevance,High Relevance,Low Relevance,Low Relevance,Low Relevance,High Relevance,High Relevance,High Relevance,High Relevance,...,Irrelevant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
import numpy as np
import pandas as pd

# Ensure both DataFrames are aligned on comparable columns
cols_compare = [f"q{i}" for i in range(1, 20)]

# Create a dictionary for faster lookup: map Origin → trainee row
trainee_dict = {trainee.loc[i, "Origin"]: trainee.iloc[i] for i in range(len(trainee))}

# Function to calculate per-row match ratio
def calculate_match(row_t, row_p):
    matches = 0
    non_empty = 0
    for c in cols_compare:
        val_t = str(row_t[c]).strip()
        val_p = str(row_p[c]).strip()
        if val_t == "" and val_p == "":
            continue
        if val_t != "" and val_p != "":
            non_empty += 1
            if val_t == val_p:
                matches += 1
    if non_empty == 0:
        return np.nan
    elif matches == non_empty:
        return 1
    else:
        return f"{matches}/{non_empty}"

# Apply only if Origins match
match_results = []
trainee_sources = []  # to store trainee's data_source_corr

for i in range(len(gpt5_renamed)):
    origin_p = gpt5_renamed.loc[i, "ID_corr"]
    if origin_p in trainee_dict:
        row_t = trainee_dict[origin_p]
        row_p = gpt5_renamed.iloc[i]
        match_results.append(calculate_match(row_t, row_p))
        trainee_sources.append(row_t["data_source_corr"])  # add trainee value
    else:
        match_results.append(np.nan)
        trainee_sources.append(np.nan)

# Add columns to physician DataFrame
gpt5_renamed["Match?"] = match_results
gpt5_renamed["data_source_corr_trainee"] = trainee_sources  # new column from trainee

# View results
gpt5_renamed[["ID_corr", "data_source_corr_trainee", "Match?"] + cols_compare].head(50)
gpt5_renamed.to_csv(paths.TABLES / "GPT5_MatchRate.csv", index=False)

In [23]:
%pip install scipy

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 25.9 MB 4.1 MB/s            
Note: you may need to restart the kernel to use updated packages.


In [25]:
import numpy as np
import pandas as pd
from scipy import stats

# Function to convert 'Match?' to numeric percentage
def match_to_percent(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float)):
        return float(x) * 100
    if isinstance(x, str) and "/" in x:
        try:
            num, denom = x.split("/")
            return (float(num) / float(denom)) * 100
        except:
            return np.nan
    return np.nan

# Create a numeric column for Match? in percentage
gpt5_renamed["Match_percent"] = gpt5_renamed["Match?"].apply(match_to_percent)

# Group by data_source_corr_trainee
grouped = gpt5_renamed.groupby("data_source_corr_trainee")["Match_percent"]

# Function to compute mean, SD, and 95% CI
def mean_sd_ci(series):
    series = series.dropna()
    n = len(series)
    if n == 0:
        return pd.Series({"mean": np.nan, "sd": np.nan, "CI_lower": np.nan, "CI_upper": np.nan, "n": 0})
    mean = series.mean()
    sd = series.std()  # standard deviation
    sem = stats.sem(series)  # standard error
    ci = sem * stats.t.ppf((1 + 0.95) / 2, n-1)  # 95% CI
    return pd.Series({"mean": mean, "sd": sd, "CI_lower": mean - ci, "CI_upper": mean + ci, "n": n})

# Apply function
summary = grouped.apply(mean_sd_ci).reset_index()

# Pivot from long to wide format
summary = summary.pivot(index='data_source_corr_trainee', columns='level_1', values='Match_percent').reset_index()

# Create beautiful formatted table
summary["Mean (%)"] = summary["mean"].apply(lambda x: f"{x:.2f}" if not pd.isna(x) else "N/A")
summary["SD (%)"] = summary["sd"].apply(lambda x: f"{x:.2f}" if not pd.isna(x) else "N/A")
summary["95% CI"] = summary.apply(
    lambda row: f"[{row['CI_lower']:.2f}, {row['CI_upper']:.2f}]" if not pd.isna(row['CI_lower']) else "N/A",
    axis=1
)
summary["N"] = summary["n"].astype(int)

# Select and rename columns for final display
result = summary[["data_source_corr_trainee", "N", "Mean (%)", "SD (%)", "95% CI"]]
result.columns = ["Data Source", "N", "Mean (%)", "SD (%)", "95% CI"]

result

,Data Source,N,Mean (%),SD (%),95% CI
0,jama,582,35.67,18.15,"[34.19, 37.14]"
1,medbullets,207,50.24,11.29,"[48.69, 51.79]"
2,medxpert,318,64.30,12.17,"[62.96, 65.64]"
3,mmlu,193,60.29,17.01,"[57.88, 62.71]"


In [26]:
# Calculate total statistics across all data sources
all_data = gpt5_renamed["Match_percent"].dropna()
n_total = len(all_data)

if n_total > 0:
    total_mean = all_data.mean()
    total_sd = all_data.std()
    total_sem = stats.sem(all_data)
    total_ci = total_sem * stats.t.ppf((1 + 0.95) / 2, n_total - 1)
    
    # Create a summary row
    total_row = pd.DataFrame({
        "Data Source": ["Total"],
        "N": [n_total],
        "Mean (%)": [f"{total_mean:.2f}"],
        "SD (%)": [f"{total_sd:.2f}"],
        "95% CI": [f"[{total_mean - total_ci:.2f}, {total_mean + total_ci:.2f}]"]
    })
    
    # Append to result table
    result = pd.concat([result, total_row], ignore_index=True)

result

,Data Source,N,Mean (%),SD (%),95% CI
0,jama,582,35.67,18.15,"[34.19, 37.14]"
1,medbullets,207,50.24,11.29,"[48.69, 51.79]"
2,medxpert,318,64.30,12.17,"[62.96, 65.64]"
3,mmlu,193,60.29,17.01,"[57.88, 62.71]"
4,Total,1300,48.65,20.06,"[47.56, 49.74]"


In [ ]:
# --- Physician Origin subset recalculation ---
from pathlib import Path
import pandas as pd
import re
import numpy as np

physician_csv = Path(r"/home/yuexing/NeuRIPS25/Physician_Labels/Mar2_2026_Data/933_Clinician_Student_Majority_Vote.csv")
physician_origins = set(
    pd.read_csv(physician_csv, usecols=['Origin'])['Origin'].astype(str).str.strip()
)

if 'df' in locals() and isinstance(df, pd.DataFrame):
    df_eval = df.copy()
elif 'output_path' in locals():
    df_eval = pd.read_csv(output_path)
else:
    raise RuntimeError('Could not find dataframe `df` or `output_path` in this notebook state.')

if 'Origin' not in df_eval.columns:
    raise KeyError('`Origin` column is missing from evaluation dataframe.')

df_eval = df_eval[df_eval['Origin'].astype(str).str.strip().isin(physician_origins)].copy()
print(f"Physician-Origin subset rows: {len(df_eval)}")

if len(df_eval) == 0:
    raise ValueError('No overlapping Origin IDs found with physician CSV.')

# Build gpt_letter when not already present
if 'gpt_letter' not in df_eval.columns:
    pred_col = None
    for c in ['gpt5_direct_prediction', 'gpt4o_direct_prediction', 'majority_vote', 'GPT5_on_72B_SR']:
        if c in df_eval.columns:
            pred_col = c
            break
    if pred_col is None:
        raise KeyError('No supported prediction column found to derive `gpt_letter`.')

    def extract_letter(x):
        if not isinstance(x, str):
            return None
        m = re.search(r'Option\s*\[?([A-J])\]?|^\s*([A-J])\s*$', str(x).strip(), flags=re.IGNORECASE)
        if m:
            return (m.group(1) or m.group(2)).upper()
        return None

    if pred_col == 'GPT5_on_72B_SR':
        df_eval['gpt_letter'] = df_eval[pred_col].astype(str).str.strip().str.upper()
    elif pred_col == 'majority_vote' and 'answer_corr' in df_eval.columns:
        # For this notebook style majority_vote is often already a letter
        df_eval['gpt_letter'] = df_eval[pred_col].astype(str).str.strip().str.upper()
    else:
        df_eval['gpt_letter'] = df_eval[pred_col].apply(extract_letter)

# Build answer_letter
if 'answer_letter' not in df_eval.columns:
    if 'answer_corr' in df_eval.columns:
        df_eval['answer_letter'] = df_eval['answer_corr'].astype(str).str.strip().str.upper()
    else:
        raise KeyError('No `answer_corr` column available to build `answer_letter`.')

# Match + binary columns
if 'gpt_letter_match' not in df_eval.columns:
    df_eval['gpt_letter_match'] = np.where(
        df_eval['gpt_letter'] == df_eval['answer_letter'],
        'Correct',
        'Incorrect'
    )

df_eval['gpt_letter_binary'] = (df_eval['gpt_letter_match'] == 'Correct').astype(int)

correct_count = int(df_eval['gpt_letter_binary'].sum())
total_count = int(df_eval['gpt_letter_binary'].notna().sum())
accuracy = correct_count / total_count if total_count > 0 else 0.0

print('\n=== Physician-Origin Recalculation ===')
print(f"Correct Predictions: {correct_count}")
print(f"Total Predictions: {total_count}")
print(f"Accuracy: {accuracy:.2%}")

source_col = None
for c in ['data_source_df3', 'data_source_corr', 'data_source_corr_trainee']:
    if c in df_eval.columns:
        source_col = c
        break

if source_col:
    print(f"\nPer-data-source stats ({source_col}):")
    for source in df_eval[source_col].dropna().unique():
        source_df = df_eval[df_eval[source_col] == source]
        c = int(source_df['gpt_letter_binary'].sum())
        t = int(source_df['gpt_letter_binary'].notna().sum())
        acc = c / t if t > 0 else 0.0
        std = source_df['gpt_letter_binary'].std(ddof=1) if t > 1 else float('nan')
        print(f"  {source}: Correct={c}, Total={t}, Accuracy={acc:.2%}, Std={std:.4f}")
